<a href="https://colab.research.google.com/github/FYasmin12/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: ranking (scoring).

My question is "which pages should a reviewer look at first?" That is not a yes/no
question for each page. It is an ordering problem, so the output is a ranked list.

To build the ranking, I use a classifier that gives each page a score for how likely
it is to be declining. Then I sort all pages by that score, highest first. So the
classifier is the tool, and the ranked list is the real output.

Why not plain classification: a yes/no label alone does not say which pages to look
at first. About 54% of pages are labelled declining, so a "yes" on its own would
cover more than half of everything and would not help a reviewer with limited time.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter file defines the target as `is_declining_label = (trend_direction == "down")`, i.e. impressions in the last 30 days fell by more than 20% versus the 30 days before. It is a proxy, not a measured "refresh paid off" outcome, because nobody in this data refreshed a page and then measured the result.

Honest limits I will state in my claims:
- It is a rule-defined label (a threshold on observed impressions), so a model can learn the rule rather than the world.
- `trend_direction` and `trend_pct` must never be features, they are the label's source (leakage).
- "Declining" does not guarantee that a refresh will fix it. Results are directional, decision-support only.

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

# Colab opened from Drive does not have the repo files, so clone the repo once
REPO_URL = "https://github.com/FYasmin12/Flyrank-ML-Internship.git"
CANDIDATES = [Path("../../data/raw/content_refresh_anonymized.csv"),
              Path("data/raw/content_refresh_anonymized.csv"),
              Path("Flyrank-ML-Internship/data/raw/content_refresh_anonymized.csv")]

if not any(p.exists() for p in CANDIDATES):
    !git clone -q {REPO_URL}

RAW = next(p for p in CANDIDATES if p.exists())

raw = pd.read_csv(RAW)
# Same filters as scripts/01_prepare_features.py
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].drop_duplicates("content_id").copy()

# Target sketch (defined by a rule on observed impressions)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(df["trend_direction"].value_counts())
print("\nbase rate (share of pages labelled declining):", round(df["is_declining_label"].mean(), 3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

base rate (share of pages labelled declining): 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: precision@K with K = 20 (the pages I would review by hand) and K = 50. It answers the editor's real question: of the pages at the top of my queue, how many truly deserve attention? Supporting metrics: recall, average precision, and a by-hand review of the top 20.

"Good" is always relative to the base rate. About 54% of pages carry the label, so a random queue already scores roughly 0.54. A model only counts as good if its precision@K clearly beats both the random base rate and a transparent baseline score.

In [8]:
import numpy as np
import pandas as pd
def precision_at_k(scores, y, k):
    top = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y)[top].mean())

# A deliberately simple, transparent baseline: visible AND stale pages first
pr = lambda s: s.rank(pct=True)
df["naive_score"] = 0.6 * pr(np.log1p(df["impressions_90d"])) + 0.4 * pr(df["days_since_last_update"])

base_rate = df["is_declining_label"].mean()
for k in (20, 50, 200):
    p = precision_at_k(df["naive_score"], df["is_declining_label"], k)
    print(f"precision@{k:<3} naive baseline = {p:.2f}   random queue ~ {base_rate:.2f}")

precision@20  naive baseline = 0.75   random queue ~ 0.54
precision@50  naive baseline = 0.56   random queue ~ 0.54
precision@200 naive baseline = 0.42   random queue ~ 0.54


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (content item), with 90-day search and analytics totals, two 30-day comparison windows, and some content properties. `content_id` is unique per row and is used only for grouping and joins; `client_id` (32 clients) is used for client-holdout splits, never as a feature.

In [9]:
print("rows x columns:", df.shape)
print("content_id unique per row:", df["content_id"].is_unique, "| clients:", df["client_id"].nunique())

show = ["content_id", "client_id", "content_type", "content_age_days", "days_since_last_update",
        "impressions_90d", "clicks_90d", "avg_position", "ctr",
        "impressions_last_30d", "impressions_prev_30d", "trend_direction", "is_declining_label"]
df[show].head(8)

rows x columns: (30000, 46)
content_id unique per row: True | clients: 32


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,avg_position,ctr,impressions_last_30d,impressions_prev_30d,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,10.6,0.76,578,987,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,20.3,0.05,2501,5915,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,36.5,0.09,2382,6089,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,6.2,0.49,3626,4206,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,44.0,0.13,4211,6452,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,8.5,0.03,617,1009,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,7.0,0.00,1,13,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,21.2,0.06,636,632,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement is either too narrow or too arbitrary here. The thresholds (180 days? 500 impressions?) are guesses, and the signals interact: the same position or content type means different things depending on the other columns. ML can learn those weights from the data, and, most importantly, I can measure whether it beats the transparent baseline using precision@K on held-out clients. If it does not beat the baseline, the honest conclusion is to use the rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.